In [373]:
import pandas as pd
import numpy as np
import datetime as dt

df_transacoes = pd.read_csv("bases/Base_de_Transacoes_e_Cupons_Capturados.csv", sep=";")
df_simulacao = pd.read_csv("bases/Base_Simulada_-_Pedestres_Av__Paulista.csv", sep=";")

df_transacoes = df_transacoes[["celular", "hora", "data"]].reset_index(drop=True)
df_transacoes["data"] = pd.to_datetime(df_transacoes["data"], dayfirst=True, errors="coerce")
df_transacoes["hora"] = pd.to_timedelta(df_transacoes["hora"].astype(str), errors="coerce")

df_simulacao = df_simulacao[df_simulacao["possui_app_picmoney"] == "Sim"]
df_simulacao = df_simulacao[["celular", "horario", "data"]].rename(columns={"horario":"hora"}).reset_index(drop=True)

df = pd.concat([df_transacoes, df_simulacao])

In [374]:
transacoes_totais = len(df_transacoes)

usuarios_unicos = len(df_transacoes["celular"].unique())

dias_analisados = len(df_transacoes["data"].unique())

print(transacoes_totais)
print(usuarios_unicos)
print(dias_analisados)

100000
4813
31


In [375]:
DAU = df_transacoes.groupby("data")["celular"].nunique()
print(DAU)
DAU = DAU.mean()
DAU = round(DAU,2)
print(f"DAU = {DAU}")

data
2025-07-01    2181
2025-07-02    2160
2025-07-03    2163
2025-07-04    2175
2025-07-05    2203
2025-07-06    2193
2025-07-07    2195
2025-07-08    2102
2025-07-09    2155
2025-07-10    2152
2025-07-11    2186
2025-07-12    2203
2025-07-13    2194
2025-07-14    2119
2025-07-15    2161
2025-07-16    2109
2025-07-17    2153
2025-07-18    2124
2025-07-19    2120
2025-07-20    2136
2025-07-21    2194
2025-07-22    2164
2025-07-23    2186
2025-07-24    2145
2025-07-25    2168
2025-07-26    2191
2025-07-27    2129
2025-07-28    2138
2025-07-29    2132
2025-07-30    2120
2025-07-31    2148
Name: celular, dtype: int64
DAU = 2158.03


In [376]:
WAU = df_transacoes.groupby(df_transacoes["data"].dt.isocalendar().week)["celular"].nunique()
print(WAU)
WAU = WAU.mean()
WAU = round(WAU, 2)
print(f"DAU = {WAU}")


week
27    4416
28    4514
29    4483
30    4510
31    4003
Name: celular, dtype: int64
DAU = 4385.2


In [377]:
MAU = df_transacoes.groupby(df_transacoes["data"].dt.month)["celular"].nunique()
print(MAU)
MAU = MAU.mean()
MAU = round(MAU, 2)
print(f"DAU = {MAU}")

data
7    4813
Name: celular, dtype: int64
DAU = 4813.0


In [378]:
data_minima = df_transacoes["data"].min()
data_limite = data_minima + pd.Timedelta(days=6)
data_retorno = data_minima + pd.Timedelta(days=30)

usuarios_iniciais = df_transacoes[df_transacoes["data"].between(data_minima, data_limite)]["celular"].unique()
usuarios_futuros = df_transacoes[df_transacoes["data"] >= data_retorno]["celular"].unique()

usuarios_retidos = set(usuarios_iniciais) & set(usuarios_futuros)
taxa_retencao = len(usuarios_retidos) / len(usuarios_iniciais)
print(f"{round(taxa_retencao*100, 2)}%")


46.2%


In [379]:
end_period = pd.Timestamp(year=df_transacoes["data"].max().year, month=7, day=31)

dfp = df_transacoes[df_transacoes["data"] <= end_period]
last_tx = dfp.groupby("celular")["data"].max().rename("ultima")
inatividade = (end_period - last_tx).dt.days.clip(lower=0)

usuarios_total = last_tx.shape[0]
usuarios_inativos = (inatividade >= 14).sum()
taxa_parada = usuarios_inativos / usuarios_total

bins = [0, 7, 14, 21, 31]
labels = ["0-7", "8-14", "15-21", "22-31"]
faixas = pd.cut(inatividade.clip(upper=31), bins=bins, labels=labels, include_lowest=True, right=True)
dist = faixas.value_counts().reindex(labels, fill_value=0).rename("qtd").to_frame()
dist["percentual"] = (dist["qtd"] / usuarios_total * 100).round(2)

print(f" Usuários inativos = {usuarios_inativos}")
print(f"usuarios totais = {usuarios_total}")
print(f"taxa churn = {round(taxa_parada*100,2)}%") 
dist.reset_index(names="faixa")

 Usuários inativos = 61
usuarios totais = 4813
taxa churn = 1.27%


,faixa,qtd,percentual
0,0-7,4567,94.89
1,8-14,194,4.03
2,15-21,33,0.69
3,22-31,19,0.39


In [380]:
# Inatividade (a partir do dist de churn)
inat_0_7      = int(dist.loc["0-7", "qtd"])
inat_8_14     = int(dist.loc["8-14", "qtd"])
inat_15_21    = int(dist.loc["15-21", "qtd"])
inat_22_31    = int(dist.loc["22-31", "qtd"])
inat_0_7_pct  = round(inat_0_7  / usuarios_total * 100, 2)
inat_8_14_pct = round(inat_8_14 / usuarios_total * 100, 2)
inat_15_21_pct= round(inat_15_21/ usuarios_total * 100, 2)
inat_22_31_pct= round(inat_22_31/ usuarios_total * 100, 2)

In [381]:

df_transacoes["timestamp"] = df_transacoes["data"].dt.normalize() + df_transacoes["hora"]
df_transacoes = df_transacoes.sort_values(["celular", "timestamp"]).copy()

df_transacoes["gap_horas"] = df_transacoes.groupby("celular")["timestamp"].diff().dt.total_seconds() / 3600
df_transacoes["nova_sessao"] = df_transacoes["gap_horas"].isna() | (df_transacoes["gap_horas"] > 4)
df_transacoes["session_id"] = df_transacoes.groupby("celular")["nova_sessao"].cumsum()

sessoes = (
    df_transacoes.groupby(["celular", "session_id"])["timestamp"]
      .agg(inicio="min", fim="max", transacoes="count")
      .reset_index()
)
sessoes["duracao_horas"] = (sessoes["fim"] - sessoes["inicio"]).dt.total_seconds() / 3600

sessoes_multiplas = sessoes[sessoes["transacoes"] > 1].shape[0]
media_duracao_horas = sessoes["duracao_horas"].mean()
segundos_totais = int(round(media_duracao_horas * 3600))
media_duracao = str(dt.timedelta(seconds=segundos_totais))

bins = [0, 0.25, 0.5, 1, 2, 4, float("inf")]
labels = ["0-15 min", "15-30 min", "30-60 min", "1-2 h", "2-4 h", "4+ h"]
sessoes["faixa"] = pd.cut(sessoes["duracao_horas"], bins=bins, labels=labels, include_lowest=True, right=True)

dist = sessoes["faixa"].value_counts().reindex(labels, fill_value=0).rename("qtd").to_frame()
dist["percentual"] = (dist["qtd"] / sessoes.shape[0] * 100).round(2)

print(f"numero de sessoes multiplas = {sessoes_multiplas}")
print(f"media de durção = {media_duracao}") 
dist.reset_index(names="faixa")

numero de sessoes multiplas = 14154
media de durção = 0:24:29


,faixa,qtd,percentual
0,0-15 min,69146,84.32
1,15-30 min,834,1.02
2,30-60 min,1669,2.04
3,1-2 h,3158,3.85
4,2-4 h,5769,7.04
5,4+ h,1425,1.74


In [382]:
# Duração de sessões (a partir do dist de duração)
dur_0_15      = int(dist.loc["0-15 min", "qtd"])
dur_15_30     = int(dist.loc["15-30 min", "qtd"])
dur_30_60     = int(dist.loc["30-60 min", "qtd"])
dur_1_2h      = int(dist.loc["1-2 h", "qtd"])
dur_2_4h      = int(dist.loc["2-4 h", "qtd"])
dur_4p        = int(dist.loc["4+ h", "qtd"])
total_sess    = sum([dur_0_15,dur_15_30,dur_30_60,dur_1_2h,dur_2_4h,dur_4p])
dur_0_15_pct  = round(dur_0_15 / total_sess * 100, 2)
dur_15_30_pct = round(dur_15_30 / total_sess * 100, 2)
dur_30_60_pct = round(dur_30_60 / total_sess * 100, 2)
dur_1_2h_pct  = round(dur_1_2h / total_sess * 100, 2)
dur_2_4h_pct  = round(dur_2_4h / total_sess * 100, 2)
dur_4p_pct    = round(dur_4p   / total_sess * 100, 2)


In [383]:
df_transacoes["timestamp"] = df_transacoes["data"].dt.normalize() + df_transacoes["hora"]
df_transacoes = df_transacoes.sort_values(["celular", "timestamp"]).copy()

df_transacoes["gap_horas"] = df_transacoes.groupby("celular")["timestamp"].diff().dt.total_seconds() / 3600
df_transacoes["nova_sessao"] = df_transacoes["gap_horas"].isna() | (df_transacoes["gap_horas"] > 4)
df_transacoes["session_id"] = df_transacoes.groupby("celular")["nova_sessao"].cumsum()

sessoes_por_usuario = df_transacoes.groupby("celular")["session_id"].nunique().reset_index(name="qtd_sessoes")

total_sessoes = int(sessoes_por_usuario["qtd_sessoes"].sum())
total_usuarios = int(sessoes_por_usuario.shape[0])
media_sessoes = round(total_sessoes / total_usuarios, 2)
top5_usuarios = sessoes_por_usuario.sort_values("qtd_sessoes", ascending=False).head(5)

bins = [0, 5, 10, 15, 20, 25, 30, np.inf]
labels = ["1-5", "6-10", "11-15", "16-20", "21-25", "26-30", "30+"]
faixas = pd.cut(sessoes_por_usuario["qtd_sessoes"], bins=bins, labels=labels, include_lowest=True, right=True)
dist_sess_user = faixas.value_counts().reindex(labels, fill_value=0).rename("qtd").to_frame()
dist_sess_user["percentual"] = (dist_sess_user["qtd"] / total_usuarios * 100).round(2)

print(total_sessoes)
print(total_usuarios)
print(media_sessoes)
dist_sess_user.reset_index(names="faixa")

82001
4813
17.04


,faixa,qtd,percentual
0,1-5,366,7.60
1,6-10,1089,22.63
2,11-15,912,18.95
3,16-20,814,16.91
4,21-25,730,15.17
5,26-30,434,9.02
6,30+,468,9.72


In [384]:
sess_1_5 = int(dist_sess_user.loc["1-5", "qtd"])
sess_6_10 = int(dist_sess_user.loc["6-10", "qtd"])
sess_11_15 = int(dist_sess_user.loc["11-15", "qtd"])
sess_16_20 = int(dist_sess_user.loc["16-20", "qtd"])
sess_21_25 = int(dist_sess_user.loc["21-25", "qtd"])
sess_26_30 = int(dist_sess_user.loc["26-30", "qtd"])
sess_30p = int(dist_sess_user.loc["30+", "qtd"])

sess_1_5_pct = float(dist_sess_user.loc["1-5", "percentual"])
sess_6_10_pct = float(dist_sess_user.loc["6-10", "percentual"])
sess_11_15_pct = float(dist_sess_user.loc["11-15", "percentual"])
sess_16_20_pct = float(dist_sess_user.loc["16-20", "percentual"])
sess_21_25_pct = float(dist_sess_user.loc["21-25", "percentual"])
sess_26_30_pct = float(dist_sess_user.loc["26-30", "percentual"])
sess_30p_pct = float(dist_sess_user.loc["30+", "percentual"])


In [385]:
t = top5_usuarios.reset_index(drop=True)
top1_fone, top1_sessoes = str(t.loc[0, "celular"]), int(t.loc[0, "qtd_sessoes"])
top2_fone, top2_sessoes = str(t.loc[1, "celular"]), int(t.loc[1, "qtd_sessoes"])
top3_fone, top3_sessoes = str(t.loc[2, "celular"]), int(t.loc[2, "qtd_sessoes"])
top4_fone, top4_sessoes = str(t.loc[3, "celular"]), int(t.loc[3, "qtd_sessoes"])
top5_fone, top5_sessoes = str(t.loc[4, "celular"]), int(t.loc[4, "qtd_sessoes"])

In [386]:
resumo_executivo = pd.DataFrame({
    "Indicador": [
        "Usuários Ativos Diários (DAU)",
        "Usuários Ativos Semanais (WAU)",
        "Usuários Ativos Mensais (MAU)",
        "Taxa de Retenção (30 dias)",
        "Taxa de Parada (Churn)",
        "Tempo Médio na Aplicação (min)",
        "Sessões por Usuário"
    ],
    "Valor": [
        DAU,
        WAU,
        MAU,
        f"{taxa_retencao}%",
        f"{taxa_parada}%",
        f"{media_duracao} min",
        f"{media_sessoes} sessões/usuario"
    ]
})

resumo_executivo


,Indicador,Valor
0,Usuários Ativos Diários (DAU),2158.03
1,Usuários Ativos Semanais (WAU),4385.2
2,Usuários Ativos Mensais (MAU),4813.0
3,Taxa de Retenção (30 dias),0.461998670507423%
4,Taxa de Parada (Churn),0.012674007895283607%
5,Tempo Médio na Aplicação (min),0:24:29 min
6,Sessões por Usuário,17.04 sessões/usuario


In [387]:
import re
from pathlib import Path

def pct_from_prop(x, nd=2):
    if x is None:
        return None
    try:
        return f"{float(x)*100:.{nd}f}\\%"
    except Exception:
        return None

def hm_from_minutes(m):
    if m is None:
        return None
    try:
        m = int(round(float(m)))
        h, r = divmod(m, 60)
        return f"{h}h {r}min" if h > 0 else f"{r}min"
    except Exception:
        return None

def val(name):
    return globals().get(name, None)

mapping = {
    "Periodo": val("periodo_str"),
    "TransacoesPeriodo": val("transacoes_totais") or val("transacoes_periodo"),
    "UsuariosUnicos": val("usuarios_unicos"),
    "DiasAnalisados": val("dias_analisados"),

    "DAUmedio": val("DAU"),
    "WAUMedio": val("WAU"),
    "MAU": val("MAU"),

    "RetencaoBase": (len(val("usuarios_iniciais")) if hasattr(val("usuarios_iniciais"), "__len__") and not isinstance(val("usuarios_iniciais"), (int, float)) else val("usuarios_iniciais")),

    "RetencaoRetornaram": (len(val("usuarios_retidos")) if isinstance(val("usuarios_retidos"), (set, list, tuple)) else val("usuarios_retidos")),
    "RetencaoTaxa": val("retencao_pct") or pct_from_prop(val("taxa_retencao")),

    "ChurnInativos": val("usuarios_inativos"),
    "ChurnTotalUsuarios": val("usuarios_total"),
    "ChurnTaxa": val("churn_pct") or pct_from_prop(val("taxa_parada")),

    "TempoMedioMin": val("media_duracao") or val("tempo_medio_min"),
    "TotalSessoesMultiolas": val("sessoes_multiplas"),
    "TotalSessoes": val("total_sessoes"),
    "TotalUsuarios": val("total_usuarios"),
    "SessoesPorUsuario": val("media_sessoes"),

    "InatA": val("inat_0_7"),
    "InatAperc": val("inat_0_7_pct") if isinstance(val("inat_0_7_pct"), str) else (f"{val('inat_0_7_pct'):.2f}\\%" if isinstance(val("inat_0_7_pct"), (int,float)) else None),
    "InatB": val("inat_8_14"),
    "InatBperc": val("inat_8_14_pct") if isinstance(val("inat_8_14_pct"), str) else (f"{val('inat_8_14_pct'):.2f}\\%" if isinstance(val("inat_8_14_pct"), (int,float)) else None),
    "InatC": val("inat_15_21"),
    "InatCperc": val("inat_15_21_pct") if isinstance(val("inat_15_21_pct"), str) else (f"{val('inat_15_21_pct'):.2f}\\%" if isinstance(val("inat_15_21_pct"), (int,float)) else None),
    "InatD": val("inat_22_31"),
    "InatDperc": val("inat_22_31_pct") if isinstance(val("inat_22_31_pct"), str) else (f"{val('inat_22_31_pct'):.2f}\\%" if isinstance(val("inat_22_31_pct"), (int,float)) else None),

    "DurZeroAQuinze": val("dur_0_15"),
    "DurZeroAQuinzePct": val("dur_0_15_pct") if isinstance(val("dur_0_15_pct"), str) else (f"{val('dur_0_15_pct'):.2f}\\%" if isinstance(val("dur_0_15_pct"), (int,float)) else None),
    "DurQuinzeATrinta": val("dur_15_30"),
    "DurQuinzeATrintaPct": val("dur_15_30_pct") if isinstance(val("dur_15_30_pct"), str) else (f"{val('dur_15_30_pct'):.2f}\\%" if isinstance(val("dur_15_30_pct"), (int,float)) else None),
    "DurTrintaASessenta": val("dur_30_60"),
    "DurTrintaASessentaPct": val("dur_30_60_pct") if isinstance(val("dur_30_60_pct"), str) else (f"{val('dur_30_60_pct'):.2f}\\%" if isinstance(val("dur_30_60_pct"), (int,float)) else None),
    "DurUmaADuasHoras": val("dur_1_2h"),
    "DurUmaADuasHorasPct": val("dur_1_2h_pct") if isinstance(val("dur_1_2h_pct"), str) else (f"{val('dur_1_2h_pct'):.2f}\\%" if isinstance(val("dur_1_2h_pct"), (int,float)) else None),
    "DurDuasAQuatroHoras": val("dur_2_4h"),
    "DurDuasAQuatroHorasPct": val("dur_2_4h_pct") if isinstance(val("dur_2_4h_pct"), str) else (f"{val('dur_2_4h_pct'):.2f}\\%" if isinstance(val("dur_2_4h_pct"), (int,float)) else None),
    "DurMaisDeQuatroHoras": val("dur_4p"),
    "DurMaisDeQuatroHorasPct": val("dur_4p_pct") if isinstance(val("dur_4p_pct"), str) else (f"{val('dur_4p_pct'):.2f}\\%" if isinstance(val("dur_4p_pct"), (int,float)) else None),

    "SessUmACinco": val("sess_1_5"),
    "SessUmACincoPct": val("sess_1_5_pct") if isinstance(val("sess_1_5_pct"), str) else (f"{val('sess_1_5_pct'):.2f}\\%" if isinstance(val("sess_1_5_pct"), (int,float)) else None),
    "SessSeisADez": val("sess_6_10"),
    "SessSeisADezPct": val("sess_6_10_pct") if isinstance(val("sess_6_10_pct"), str) else (f"{val('sess_6_10_pct'):.2f}\\%" if isinstance(val("sess_6_10_pct"), (int,float)) else None),
    "SessOnzeAQuinze": val("sess_11_15"),
    "SessOnzeAQuinzePct": val("sess_11_15_pct") if isinstance(val("sess_11_15_pct"), str) else (f"{val('sess_11_15_pct'):.2f}\\%" if isinstance(val("sess_11_15_pct"), (int,float)) else None),
    "SessVinteEUmAVinteECinco": val("sess_21_25"),
    "SessVinteEUmAVinteECincoPct": val("sess_21_25_pct") if isinstance(val("sess_21_25_pct"), str) else (f"{val('sess_21_25_pct'):.2f}\\%" if isinstance(val("sess_21_25_pct"), (int,float)) else None),
    "SessVinteESeisATrinta": val("sess_26_30"),
    "SessVinteESeisATrintaPct": val("sess_26_30_pct") if isinstance(val("sess_26_30_pct"), str) else (f"{val('sess_26_30_pct'):.2f}\\%" if isinstance(val("sess_26_30_pct"), (int,float)) else None),
    "SessMaisDeTrinta": val("sess_30p"),
    "SessMaisDeTrintaPct": val("sess_30p_pct") if isinstance(val("sess_30p_pct"), str) else (f"{val('sess_30p_pct'):.2f}\\%" if isinstance(val("sess_30p_pct"), (int,float)) else None),

    "TopAFone": val("top1_fone"),
    "TopASessoes": val("top1_sessoes"),
    "TopBFone": val("top2_fone"),
    "TopBSessoes": val("top2_sessoes"),
    "TopCFone": val("top3_fone"),
    "TopCSessoes": val("top3_sessoes"),
    "TopDFone": val("top4_fone"),
    "TopDSessoes": val("top4_sessoes"),
    "TopEFone": val("top5_fone"),
    "TopESessoes": val("top5_sessoes"),
}

def replace_macro(text, macro, value):
    if value is None:
        return text
    value_str = str(value)
    pat = re.compile(rf"(\\renewcommand\{{\\{re.escape(macro)}\}}\{{)(.*?)(\}})", re.DOTALL)
    return pat.sub(lambda m: m.group(1) + value_str + m.group(3), text)

tex_path = Path("relatorio/valores.tex")
src = tex_path.read_text(encoding="utf-8")

for macro, value in mapping.items():  # mapping = { "TransacoesPeriodo": transacoes_totais, ... }
    src = replace_macro(src, macro, value)

tex_path.write_text(src, encoding="utf-8")
print("valores.tex atualizado.")


valores.tex atualizado.
